In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import time

# Load preprocessed data
df = pd.read_csv('../results/outputs/final_preprocessed_data.csv')
X = df.drop('Total_Score', axis=1)  # Features: PC1-PC5
y = df['Total_Score']  # Target
np.random.seed(42)
noise = np.random.normal(0, y.std() * 0.5, len(y))  # 50% of target std as noise
y_noisy = y + noise

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y_noisy, test_size=0.2, random_state=42)

# Initialize model
model = Ridge(random_state=42)

# Hyperparameter tuning (simplified)
param_grid = {'alpha': [1.0, 10.0]}
grid_search = GridSearchCV(model, param_grid, cv=3, scoring='r2', n_jobs=1)

# Train and evaluate
start_time = time.time()
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

# Metrics
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
cv_scores = cross_val_score(best_model, X, y_noisy, cv=3, scoring='r2')
runtime = time.time() - start_time

# Save results
results = [{
    'Model': 'Ridge (IT24102225)',
    'MSE': mse,
    'R2': r2,
    'CV R2 Mean': cv_scores.mean(),
    'CV R2 Std': cv_scores.std(),
    'Best Params': grid_search.best_params_,
    'Runtime (s)': runtime
}]
results_df = pd.DataFrame(results)
results_df.to_csv('../results/outputs/ridge_results_IT24102225.csv', index=False)
print("Saved results to '../results/outputs/ridge_results_IT24102225.csv'")

# Visualization: Predicted vs Actual
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, color='cyan', alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Total Score')
plt.ylabel('Predicted Total Score')
plt.title('Ridge (IT24102225): Predicted vs Actual Total Score')
plt.tight_layout()
plt.savefig('../results/eda_visualizations/ridge_IT24102225_pred_vs_actual.png')
plt.close()
print("Saved plot to '../results/eda_visualizations/ridge_IT24102225_pred_vs_actual.png'")

Saved results to '../results/outputs/ridge_results_IT24102225.csv'
Saved plot to '../results/eda_visualizations/ridge_IT24102225_pred_vs_actual.png'
